In [1]:
import pyspark
from pyspark.sql import SparkSession

In [4]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [5]:
from pyspark.sql import types

In [12]:
yellow_schema = types.StructType([
types.StructField('VendorID', types.IntegerType(), True), 
types.StructField('tpep_pickup_datetime', types.TimestampType(), True), 
types.StructField('tpep_dropoff_datetime', types.TimestampType(), True), 
types.StructField('passenger_count', types.IntegerType(), True), 
types.StructField('trip_distance', types.DoubleType(), True), 
types.StructField('RatecodeID', types.IntegerType(), True), 
types.StructField('store_and_fwd_flag', types.StringType(), True), 
types.StructField('PULocationID', types.IntegerType(), True), 
types.StructField('DOLocationID', types.IntegerType(), True), 
types.StructField('payment_type', types.IntegerType(), True), 
types.StructField('fare_amount', types.DoubleType(), True), 
types.StructField('extra', types.DoubleType(), True), 
types.StructField('mta_tax', types.DoubleType(), True), 
types.StructField('tip_amount', types.DoubleType(), True), 
types.StructField('tolls_amount', types.DoubleType(), True), 
types.StructField('improvement_surcharge', types.DoubleType(), True), 
types.StructField('total_amount', types.DoubleType(), True), 
types.StructField('congestion_surcharge', types.DoubleType(), True), 
types.StructField('Airport_fee', types.DoubleType(), True), 
types.StructField('cbd_congestion_fee', types.DoubleType(), True)
])

In [6]:
df_yellow = spark.read \
    .parquet('yellow_tripdata_2025-11.parquet')

In [8]:
df_yellow.schema

StructType([StructField('VendorID', IntegerType(), True), StructField('tpep_pickup_datetime', TimestampNTZType(), True), StructField('tpep_dropoff_datetime', TimestampNTZType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('Airport_fee', DoubleType(), True), StructField('cbd_congestio

In [9]:
 path = 'y_tripdata_2025-11'

In [10]:
df_yellow \
        .repartition(4) \
        .write.parquet(path, mode='overwrite')

In [11]:
df_yellow = spark.read.parquet('y_tripdata_2025-11/')

In [12]:
df_yellow.select('VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'trip_distance').show(10)

+--------+--------------------+---------------------+-------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|
+--------+--------------------+---------------------+-------------+
|       2| 2025-11-07 18:37:45|  2025-11-07 18:41:51|         0.78|
|       2| 2025-11-07 23:14:45|  2025-11-08 00:10:08|         18.2|
|       2| 2025-11-01 22:40:45|  2025-11-01 22:44:52|         0.44|
|       1| 2025-11-07 13:57:39|  2025-11-07 14:19:11|          7.6|
|       2| 2025-11-02 18:02:18|  2025-11-02 18:37:23|          4.2|
|       2| 2025-11-05 09:05:48|  2025-11-05 09:18:11|         0.94|
|       2| 2025-11-09 18:01:11|  2025-11-09 18:17:31|          1.1|
|       2| 2025-11-08 11:20:15|  2025-11-08 11:26:17|         0.46|
|       2| 2025-11-02 10:32:27|  2025-11-02 10:41:20|         2.11|
|       1| 2025-11-08 13:25:10|  2025-11-08 13:43:13|          2.6|
+--------+--------------------+---------------------+-------------+
only showing top 10 rows


In [24]:
df_yellow.registerTempTable('trips_data')

/home/sxzquare/my-docker-workbook/06-batch/.venv/lib/python3.11/site-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [25]:
spark.sql("""
SELECT
    count(1)
FROM
    trips_data
WHERE
    DATE(tpep_pickup_datetime) = '2025-11-15'
""").show()

[Stage 11:=============================>                            (2 + 2) / 4]

+--------+
|count(1)|
+--------+
|  162604|
+--------+



In [54]:
spark.sql("""
SELECT ROUND(
    MAX(
        (UNIX_TIMESTAMP(tpep_dropoff_datetime) -
         UNIX_TIMESTAMP(tpep_pickup_datetime)) / 3600
    ), 1
) AS longest_trip_hours
FROM trips_data
""").show()

[Stage 58:===========================================>              (3 + 1) / 4]

+------------------+
|longest_trip_hours|
+------------------+
|              90.6|
+------------------+



In [65]:
!mkdir -p /taxi_zone

mkdir: cannot create directory ‘/taxi_zone’: Permission denied


In [66]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv -O ./taxi_zone/taxi_zone_lookup.csv

/taxi_zone/taxi_zone_lookup.csv: No such file or directory


In [17]:
df_zones = spark.read \
    .option("header", True) \
    .csv("taxi_zone/taxi_zone_lookup.csv", inferSchema=True)

df_zones.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [18]:
df_zones.printSchema()

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [21]:
df_zones.createOrReplaceTempView('taxi_zones')

In [29]:
spark.sql("""
SELECT 
    z.Zone,
    y.PULocationID, 
    COUNT(1) as count
FROM trips_data y
JOIN taxi_zones z
      ON y.PULocationID = z.LocationID
GROUP BY z.Zone, y.PULocationID
ORDER BY count
LIMIT 5
    """).show()


[Stage 27:=============================>                            (2 + 2) / 4]

+--------------------+------------+-----+
|                Zone|PULocationID|count|
+--------------------+------------+-----+
|       Arden Heights|           5|    1|
|Governor's Island...|         105|    1|
|Eltingville/Annad...|          84|    1|
|       Port Richmond|         187|    3|
| Green-Wood Cemetery|         111|    4|
+--------------------+------------+-----+

